In [0]:
%sql
create schema if not exists cyntexa_dev.logistics;
Create volume if not exists cyntexa_dev.logistics.raw;

## Task 1:
Use CTAS with read_files() to ingest a CSV file into a managed Delta table.

In [0]:
%sql
Create table cyntexa_dev.logistics.logistics_raw as
select *
from read_files(
    '/Volumes/cyntexa_dev/logistics/raw/logistics_2026_08_18.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

In [0]:
%sql
select * from cyntexa_dev.logistics.logistics_raw limit 10;

## Task 2:
 Ingest a nested JSON file, extracting at least 2 nested fields into top-level columns. 

In [0]:
%sql
Create table if not exists cyntexa_dev.logistics.logistics_raw_json
using delta as
select
    shipment_id,
    ship_date,
    carrier,
    total_weight_kg,
    customer.name as customer_name,
    customer.contact.email as customer_email,
    customer.contact.phone as customer_phone,
    items
From read_files(
    '/Volumes/cyntexa_dev/logistics/raw/logistics_customers_2026_08_18.json',
    format => 'json'
)

In [0]:
%sql
select * from cyntexa_dev.logistics.logistics_raw_json limit 10

## Task 3:
Run DESCRIBE, DESCRIBE EXTENDED, and DESCRIBE DETAIL on your new table and note what unique information each one gives you. 


In [0]:
%sql
Describe table cyntexa_dev.logistics.logistics_raw

In [0]:
%sql
Describe extended cyntexa_dev.logistics.logistics_raw;

In [0]:
%sql
Describe detail cyntexa_dev.logistics.logistics_raw

**Describe:** This just gives the structure of the schemma of the table.

**Describe Extended:** It shows everything the _Describe_ shows plus the metadata (like catalog, database, owner, created time, table type, storage location and table properties).

**Describe Detail:** A single structured row with format, uuid, location, number of parquet files, lastnmodiefied

## Test 4:
Add _metadata.file_name and _metadata.file_path to your ingestion query and use them to prove which source file each row came from. 


In [0]:
%sql
Create table if not exists cyntexa_dev.logistics.logistics_raw_metadata as
Select *,
    _metadata.file_name as file_name,
    _metadata.file_path as file_path,
    current_timestamp() as ingestion_date
From read_files (
    '/Volumes/cyntexa_dev/logistics/raw/logistics_2026_08_18.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)

In [0]:
%sql
select * from cyntexa_dev.logistics.logistics_raw_metadata limit 10

## Task 5:
Create an Iceberg table from the same source data and compare its DESCRIBE DETAIL output (format,
location) to the Delta version.

In [0]:
%sql
Create table if not exists cyntexa_dev.logistics.logistics_raw_iceberg
using iceberg as 
select *,
    _metadata.file_name as file_name,
    _metadata.file_path as file_path,
    current_timestamp() as ingestion_date
From read_files (
    '/Volumes/cyntexa_dev/logistics/raw/logistics_2026_08_18.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);
    
select * from cyntexa_dev.logistics.logistics_raw_iceberg limit 10

In [0]:
%sql
Describe Detail cyntexa_dev.logistics.logistics_raw_iceberg

## Task 6:
(Data Analyst) Write a query using the metadata columns to build a 'records per source file' audit
report — useful for verifying a vendor's daily file drop.

In [0]:
%sql
-- Inserting to the metadata table first from a different csv file
Insert Into cyntexa_dev.logistics.logistics_raw_metadata
Select *,
    _metadata.file_name as file_name,
    _metadata.file_path as file_path,
    current_timestamp() as ingestion_date
From read_files (
    '/Volumes/cyntexa_dev/logistics/raw/logistics_2026_08_19.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)

In [0]:
%sql
Select
    count(*) as total_record,
    file_name,
    file_path
From cyntexa_dev.logistics.logistics_raw_metadata
group by file_name, file_path
order by total_record

## Task 7:
Land multiple CSV files with slightly different formats (e.g., an extra column, a different delimiter) and document how your read_files() options need to change for each, plus how you'd detect a mismatch before it silently breaks downstream reports. 

In [0]:
%sql
Create or replace table cyntexa_dev.logistics.logistics_raw_e as
Select *,
    _metadata.file_name as file_name,
    _metadata.file_path as file_path,
    current_timestamp() as ingestion_date
From read_files (
    '/Volumes/cyntexa_dev/logistics/raw/logistics/',
    format => 'csv',
    header => true,
    schema => 'shipment_id STRING, origin_city STRING, destination_city STRING, carrier STRING, status STRING, ship_date DATE, weight_kg DOUBLE',
    rescuedDataColumn => 'rescuedData'
)

In [0]:
%sql
select * from cyntexa_dev.logistics.logistics_raw_e

To read multiple csv files with slightly different schema we can use just read_files, this read_files function reads all the schema of those files and, as I have three files one with an extra column `priority`, the table that was created will have the extra column and for the data of the first file the column value of payment_method will be null, and in case of different delimiters we can add sep => ',' or '|' inside the read_files, so for that we can also handle that.

But other than the extra column read_files also create another column _rescued_data which is very usefull for checking the mismatch that was there in the downstream reports.

the _rescuse_data column helps rescues data in case any problematic data appears then that engine put that data in the _rescuse_data column. The _rescuse_data data column rescues data in case of extra column, different data type and the column name miss match, so instead of droping the data entierly and putting null value, the engine puts the null value in the original column and put the orignal bad data in the _rescuse_data column.

## Task 8:
Write a decision memo: when should Cyntexa choose Iceberg (or Delta UniForm) over native Delta
for a given table, considering downstream tools like Snowflake or Trino?

Cyntexa should choose Iceberg over the native delta for a given table, for the following reasons:

- Iceberg seamlessly works with Apache Spark, Trino, and Apache flink, and major cloud offers support offer native supports. Delta by contrast is build around the Apache Ecosystem and databricks ecosystem. So if we need the long term flexiblity around different query engine, we should go with `Iceberg` but if we are heaviely invested in spark/databricks we should go with `Delta`.
- If Cyntexa's vendor files are likely to change the schema over the time, we should go with `Iceberg` as it provides very strong Schema Evolution capablities.
- For very large tables, Iceberg's partition evolution feature lets you change how data is organized without downtime or expensive migrations, alongside general scalability to manage billions of files and petabytes of data.
- Icerberg also provides the time travel capablities using the snapshots.

## Task 9:
Use DESCRIBE HISTORY together with the metadata columns to trace a specific bad row back to the
exact ingestion run and source file that introduced it.

In [0]:
%sql
-- Adding the new file to the metadata table with some bad values

Insert into table cyntexa_dev.logistics.logistics_raw_metadata
Select *,
    _metadata.file_name AS source_file_name,
    _metadata.file_path AS source_file_path,
    current_timestamp() AS ingestion_date
FROM read_files(
    '/Volumes/cyntexa_dev/logistics/raw/shipments_2026_08_18.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)

In [0]:
%sql

-- Inserting the second day 
Insert into table cyntexa_dev.logistics.logistics_raw_metadata
Select *,
    _metadata.file_name AS source_file_name,
    _metadata.file_path AS source_file_path,
    current_timestamp() AS ingestion_date
FROM read_files(
    '/Volumes/cyntexa_dev/logistics/raw/shipments_2026_08_19.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)

In [0]:
%sql
Describe History cyntexa_dev.logistics.logistics_raw_metadata

In [0]:
%sql

-- Checking for the bad records
Select * from cyntexa_dev.logistics.logistics_raw_metadata
Where weight_kg < 0 or carrier is Null or carrier = '' or _rescued_data is not null

In [0]:
%sql
-- Getting the ingestion date
SELECT shipment_id, weight_kg, carrier, file_name, file_path, ingestion_date
FROM cyntexa_dev.logistics.logistics_raw_metadata
WHERE shipment_id = 'SHP3003';

In [0]:
%sql
SELECT shipment_id, weight_kg, carrier, file_name, file_path, ingestion_date
FROM cyntexa_dev.logistics.logistics_raw_metadata
WHERE shipment_id = 'SHP3008';

The bad row for 'SHP3003' was introduced in the `Write` of the version 3 and timestamp `2026-08-25T05:51:45.000+00:00`, the source was `shipments_2026_08_19.csv`

The bad row for 'SHP3008' was introduced in the `Write` of the version 3 and timestamp `2026-08-25T05:51:45.000+00:00`, the source was `shipments_2026_08_19.csv`

So from the above select query I can check for the bad values like nulls and negative values, and also the metadata and comparet the metadat of bad files with the metadata given by `Describe history` I can get the exact version of the table from where I got the bad data and can go back to the previous good version.